# 08 · End-to-End Encrypted Inference

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sabrinaaquino/base-batches-workshop/blob/main/notebooks/08-e2ee-encryption.ipynb)

This is the strongest privacy mode Venice offers. Your prompt is encrypted **on your machine** before it leaves, stays encrypted as it passes through Venice's infrastructure, and is **only decrypted inside a hardware-secured enclave** (TEE) on the GPU.

By the end of this notebook you will:
1. Send the same prompt twice — once to a regular model, once to an E2EE model — and **see the difference on the wire**
2. Implement the full E2EE protocol from scratch in pure Python
3. Verify the TEE attestation on the response cryptographically

The point: you don't have to *trust* Venice. You can *verify* it.

**Crypto used:**
- ECDH on **secp256k1** for key exchange
- **HKDF-SHA256** for key derivation
- **AES-256-GCM** for symmetric encryption

In [ ]:
%pip install --quiet openai requests python-dotenv rich coincurve cryptography eth-account

In [ ]:
import os, sys

# Try Colab secrets first
try:
    from google.colab import userdata  # type: ignore
    api_key = userdata.get("VENICE_API_KEY")
    os.environ["VENICE_API_KEY"] = api_key
except Exception:
    api_key = os.environ.get("VENICE_API_KEY")

if not api_key:
    from getpass import getpass
    api_key = getpass("Paste your Venice API key: ").strip()
    os.environ["VENICE_API_KEY"] = api_key

from openai import OpenAI
client = OpenAI(
    api_key=api_key,
    base_url="https://api.venice.ai/api/v1",
)
print("Connected to Venice ✔")

## 1. Pick an E2EE model

Filter the model list for ones that support E2EE.

In [ ]:
import requests, json

models = requests.get(
    "https://api.venice.ai/api/v1/models",
    headers={"Authorization": f"Bearer {api_key}"},
).json()

e2ee_models = []
tee_models = []
for m in models["data"]:
    caps = (m.get("model_spec", {}) or {}).get("capabilities", {}) or {}
    if caps.get("supportsE2EE"):
        e2ee_models.append(m["id"])
    if caps.get("supportsTeeAttestation"):
        tee_models.append(m["id"])

print("E2EE models:", e2ee_models[:5])
print("TEE models: ", tee_models[:5])

# Pick one. Models prefixed with `e2ee-` are encrypted; `tee-` are
# enclave-isolated but not client-encrypted.
E2EE_MODEL = e2ee_models[0] if e2ee_models else "e2ee-qwen3-5-122b-a10b"
print(f"\nUsing: {E2EE_MODEL}")

## 2. The control: send a regular prompt

Look at the request body that leaves your machine. It's plaintext.

In [ ]:
payload = {
    "model": "venice-uncensored",
    "messages": [{"role": "user", "content": "MY SECRET: I am building a regret vending machine."}],
}
print("📤 Plaintext request body:")
print(json.dumps(payload, indent=2))

# Anyone sniffing the wire (or with access to a logging proxy) sees the secret.

## 3. The E2EE flow — step by step

We'll do it manually so you can see every move.

### 3a. Generate an ephemeral keypair (secp256k1)

Throwaway. New one per session — never reuse.

In [ ]:
from coincurve import PrivateKey

ephemeral = PrivateKey()
ephemeral_pub = ephemeral.public_key.format(compressed=False).hex()  # 04 + X + Y
print(f"Ephemeral public key (uncompressed hex): {ephemeral_pub}")
print(f"Length: {len(ephemeral_pub)} hex chars  (1 + 32 + 32 = 65 bytes)")

### 3b. Fetch the TEE's public key from its attestation

The enclave publishes a public key that's **bound to the hardware** — it can only have been generated inside the genuine enclave.

In [ ]:
attest = requests.get(
    f"https://api.venice.ai/api/v1/models/{E2EE_MODEL}/attestation",
    headers={"Authorization": f"Bearer {api_key}"},
).json()

print("Attestation summary:")
for k in ("verified", "signing_address", "encryption_pub_key", "nonce", "provider"):
    if k in attest:
        v = attest[k]
        if isinstance(v, str) and len(v) > 80:
            v = v[:80] + "..."
        print(f"  {k}: {v}")

tee_pub_hex = attest["encryption_pub_key"]
print(f"\n✓ TEE will be using public key: {tee_pub_hex[:32]}...")

### 3c. Derive a shared symmetric key

ECDH gives us a shared secret only our ephemeral private key + the TEE's private key (inside the enclave) can compute. Pipe it through HKDF to get a clean AES-256 key.

In [ ]:
from coincurve import PublicKey
from cryptography.hazmat.primitives.kdf.hkdf import HKDF
from cryptography.hazmat.primitives import hashes

tee_pub = PublicKey(bytes.fromhex(tee_pub_hex))
shared = ephemeral.ecdh(tee_pub.format(compressed=False))  # raw 32-byte secret

aes_key = HKDF(
    algorithm=hashes.SHA256(),
    length=32,
    salt=None,
    info=b"venice-e2ee-v1",
).derive(shared)

print(f"Shared secret (raw):  {shared.hex()[:40]}...")
print(f"Derived AES-256 key: {aes_key.hex()[:40]}...")

### 3d. Encrypt the prompt with AES-256-GCM

Venice uses a **32-byte nonce** (unusual — most AES-GCM uses 12). Don't trim it.

In [ ]:
import os
from cryptography.hazmat.primitives.ciphers.aead import AESGCM

NONCE_LEN = 32  # Venice TEE requirement

plaintext_messages = [{"role": "user", "content": "MY SECRET: I am building a regret vending machine."}]
plaintext_bytes = json.dumps(plaintext_messages).encode()

aesgcm = AESGCM(aes_key)
nonce = os.urandom(NONCE_LEN)
ciphertext = aesgcm.encrypt(nonce, plaintext_bytes, associated_data=None)

print(f"Plaintext: {plaintext_bytes!r}")
print(f"Ciphertext ({len(ciphertext)} bytes): {ciphertext.hex()[:80]}...")
print(f"Nonce: {nonce.hex()}")

### 3e. Send the ciphertext to Venice

Notice — **nothing in this request body is human-readable**. Venice's logs would see ciphertext only.

In [ ]:
e2ee_payload = {
    "model": E2EE_MODEL,
    "ciphertext": ciphertext.hex(),
    "nonce": nonce.hex(),
    "ephemeral_pub_key": ephemeral_pub,
    # Optional metadata that's safe to send in plaintext
    "max_tokens": 256,
}

print("📤 E2EE request body:")
print(json.dumps({k: (v[:60] + '...' if isinstance(v, str) and len(v) > 60 else v) for k, v in e2ee_payload.items()}, indent=2))

r = requests.post(
    "https://api.venice.ai/api/v1/chat/completions",
    headers={"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"},
    json=e2ee_payload,
)
print(f"\nStatus: {r.status_code}")
response = r.json()
print(json.dumps(response, indent=2)[:600])

### 3f. Decrypt the response

The TEE encrypted its reply using the same shared secret.

In [ ]:
reply_ciphertext = bytes.fromhex(response["ciphertext"])
reply_nonce = bytes.fromhex(response["nonce"])

reply_plaintext = aesgcm.decrypt(reply_nonce, reply_ciphertext, associated_data=None)
print("📥 Decrypted reply:")
print(json.loads(reply_plaintext))

## 4. Verify the response was actually signed by the enclave

Every E2EE response includes a signature from a key that **only exists inside the enclave**. We check the signature against the `signing_address` from the attestation.

In [ ]:
from eth_account.messages import encode_defunct
from eth_account import Account

signing_address = attest["signing_address"]
signature = response["signature"]
signed_payload = response["signed_payload"]  # the bytes the enclave actually signed

recovered = Account.recover_message(
    encode_defunct(text=signed_payload),
    signature=signature,
)

if recovered.lower() == signing_address.lower():
    print(f"✓ Signature verified.")
    print(f"  Signer: {recovered}")
    print(f"  Matches enclave address from attestation: {signing_address}")
else:
    print(f"✗ MISMATCH. Recovered {recovered}, expected {signing_address}")

## 5. Tamper test — flip one byte

Modify a single character of the response and the verification should immediately fail.

In [ ]:
tampered = signed_payload[:-1] + ("X" if signed_payload[-1] != "X" else "Y")

recovered = Account.recover_message(
    encode_defunct(text=tampered),
    signature=signature,
)

if recovered.lower() == signing_address.lower():
    print("This should not happen — tampering went undetected.")
else:
    print(f"✓ Tampering detected. Recovered {recovered} ≠ enclave {signing_address}")

## 6. The bounty (try this with the room)

Take the ciphertext from cell 3d above and post it publicly — Discord, Twitter, anywhere. Offer a prize to anyone who can recover the plaintext.

You're safe. Without **either** your ephemeral private key (which only exists in this notebook's memory and is gone when the kernel dies) **or** the TEE's private key (which only exists inside the hardware enclave and was never extracted), **the math does not allow** decryption.

```python
print(f"Public bounty:")
print(f"  ciphertext: {ciphertext.hex()}")
print(f"  nonce:      {nonce.hex()}")
print(f"  ephemeral:  {ephemeral_pub}")
print(f"  TEE pubkey: {tee_pub_hex}")
```

That's all the public information. The only secret left in the universe that can decrypt this is inside an Intel TDX / NVIDIA Confidential Compute enclave that even Venice's engineers can't read.

## What just happened

You implemented end-to-end encrypted AI inference from primitives:

1. **You generated** a fresh secp256k1 keypair locally.
2. **You verified** Venice gave you a public key that came from inside a hardware enclave.
3. **You derived** an AES key that only your ephemeral key + that enclave can produce.
4. **You encrypted** your prompt before it left your laptop.
5. **The enclave** decrypted, ran the model, encrypted the reply, signed it.
6. **You verified** the signature came from inside the enclave.
7. **You tampered** with the response and watched the verification break.

Every other AI workshop today asks you to *trust* the provider.

This notebook just made you a **verifier**.

---

## Want to run agents on this?

The full set of `e2ee-*` and `tee-*` models is at `https://api.venice.ai/api/v1/models`. The same flow above wires into LangChain, the Vercel AI SDK, CrewAI, or anything that accepts a custom HTTP client.

For production, use [`venice-x402-client`](https://www.npmjs.com/package/venice-x402-client) (TypeScript) or hand-roll the equivalent in Python — it's about 100 lines once factored.

**You're done. Welcome to the Venice ecosystem.**